In [ ]:
# install dependencies
# !pip install -q --upgrade git+https://github.com/terrierteam/pyterrier_adaptive.git
# !pip install -q --upgrade git+https://github.com/terrierteam/pyterrier_t5.git
# !pip install -q pyterrier_pisa

In [1]:
# imports
import pyterrier as pt
if not pt.started(): 
    pt.init()
from pyterrier.measures import *
from pyterrier_adaptive import GAR, CorpusGraph
from pyterrier_t5 import MonoT5ReRanker
from pyterrier_pisa import PisaIndex

/home/peppe/anaconda3/envs/GNRR/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTerrier 0.10.0 has loaded Terrier 5.8 (built by craigm on 2023-11-01 18:05) and terrier-helper 0.0.8

No etc/terrier.properties, using terrier.default.properties for bootstrap configuration.


In [10]:
# Create required components
dataset = pt.get_dataset('irds:msmarco-passage')
retriever = PisaIndex.from_dataset('msmarco_passage').bm25()
scorer = pt.text.get_text(dataset, 'text') >> MonoT5ReRanker(verbose=False, batch_size=16)
graph = CorpusGraph.from_dataset('msmarco_passage', 'corpusgraph_bm25_k16').to_limit_k(8)

/home/peppe/anaconda3/envs/GNRR/lib/python3.8/site-packages/transformers/models/t5/tokenization_t5.py:240: FutureWarning: This tokenizer was incorrectly instantiated with a model max length of 512 which will be corrected in Transformers v5.
For now, this behavior is kept to avoid breaking backwards compatibility when padding/encoding with `truncation is True`.
- Be aware that you SHOULD NOT rely on t5-base automatically truncating your input to 512 when padding/encoding.
- If you want to encode/pad to sequences longer than 512 you can either instantiate this tokenizer with `model_max_length` or pass `max_length` when encoding/padding.
- To avoid this warning, please instantiate this tokenizer with `model_max_length` set to your preferred value.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:
# A simple example
pipeline = retriever >> GAR(scorer, graph) >> pt.text.get_text(dataset, 'text')

pipeline.search('clustering hypothesis information retrieval')

In [11]:
dataset = pt.get_dataset('irds:msmarco-passage/trec-dl-2019/judged')
pt.Experiment(
    [retriever, retriever >> scorer, retriever >> GAR(scorer, graph)],
    dataset.get_topics(),
    dataset.get_qrels(),
    [nDCG, MAP(rel=2), R(rel=2)@1000],
    names=['bm25', 'bm25 >> monot5', 'bm25 >> GAR(monot5)']
)

[INFO] [starting] building docstore
docs_iter: 100%|█████████████████| 8841823/8841823 [00:45<00:00, 194082.44doc/s]
[INFO] [finished] docs_iter: [00:45] [8841823doc] [194081.30doc/s]
[INFO] [finished] building docstore [45.56s]


,name,nDCG,AP(rel=2),R(rel=2)@1000
0,bm25,0.602325,0.303099,0.755495
1,bm25 >> monot5,0.696611,0.482574,0.755495
2,bm25 >> GAR(monot5),0.724549,0.490025,0.825952


In [9]:
dataset = pt.get_dataset('irds:msmarco-passage/trec-dl-2019/judged')

dataset.get_qrels()

,qid,docno,label,iteration
0,19335,1017759,0,Q0
1,19335,1082489,0,Q0
2,19335,109063,0,Q0
3,19335,1160863,0,Q0
4,19335,1160871,0,Q0
...,...,...,...,...
9255,1133167,8839920,2,Q0
9256,1133167,8839922,2,Q0
9257,1133167,944810,0,Q0
9258,1133167,949411,0,Q0
